# UmarTransit-1B: Export to GGUF Format

Convert the fine-tuned model to GGUF for local inference with Ollama/llama.cpp.

**Input:** `umarfarookm/UmarTransit-1B` (BF16 on HuggingFace)
**Output:** GGUF files (q4_k_m + q8_0) pushed to same HF repo

> **Requirements:** Google Colab with T4 GPU runtime.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Configuration

In [ ]:
MODEL_ID = "umarfarookm/UmarTransit-1B"
MAX_SEQ_LENGTH = 1024
OUTPUT_DIR = "./gguf-output"

# GGUF quantization types to export
QUANT_METHODS = [
    "q4_k_m",  # 4-bit, ~1 GB, recommended for most use cases
    "q8_0",    # 8-bit, ~1.6 GB, higher quality
]

## 3. Load Model from HuggingFace

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=False,  # Load full precision for GGUF conversion
)

print(f"Model loaded: {MODEL_ID}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## 4. Export to GGUF

Save GGUF files locally in two quantization levels.

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

for method in QUANT_METHODS:
    print(f"\nExporting GGUF with quantization: {method}")
    print("=" * 50)
    model.save_pretrained_gguf(
        OUTPUT_DIR,
        tokenizer,
        quantization_method=method,
    )
    print(f"Done: {method}")

# Show exported files
print("\nExported GGUF files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".gguf"):
        size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024 / 1024
        print(f"  {f}: {size_mb:.1f} MB")

## 5. Push GGUF to HuggingFace

Upload GGUF files to the same model repo.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()

# Upload all GGUF files to the model repo
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".gguf"):
        filepath = os.path.join(OUTPUT_DIR, f)
        size_mb = os.path.getsize(filepath) / 1024 / 1024
        print(f"Uploading {f} ({size_mb:.1f} MB)...")
        api.upload_file(
            path_or_fileobj=filepath,
            path_in_repo=f,
            repo_id=MODEL_ID,
        )
        print(f"  Uploaded: https://huggingface.co/{MODEL_ID}/blob/main/{f}")

print(f"\nAll GGUF files uploaded to: https://huggingface.co/{MODEL_ID}")
print("Done! You can now use these with Ollama or llama.cpp.")